# Setup

In [ ]:
from pprint import pprint
from tempfile import TemporaryDirectory
from pathlib import Path


# Seed
seed = 42


# Workspace
workspace = TemporaryDirectory(     # temporary create a folder.
    prefix='ai_summary_prac_',
)
source_path = Path(workspace.name) / 'average.py'

print(f"Target source path: {source_path}")

Target source path: C:\Users\yana\AppData\Local\Temp\ai_summary_prac_s8zz_kpm\average.py


# 1. Terminologies

- Trajectory: a collection of observations and actions for one task.
- Rollout: a trajectory collection for agent learning.

# 2. Rollout

In [ ]:
import json
from math import isclose, isfinite
from pathlib import Path
from uuid import uuid4

from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain.messages import SystemMessage, HumanMessage, ToolMessage
from langchain.tools import tool


# 1. Task and model
buggy_code = "def average(values):\    return sum(values) // len(values)\n"
tests = [
    ([], 0.0),
    ([0], 0.0),
    ([1, 2], 1.5),
    ([-2, -3], -2.5),
    ([0.1, 0.2, 0.3], 0.2),
]

model_config = dict(
    model='qwen3:4b',
    temperature=0,
    reasoning=False,
    seed=42,
)
chat = ChatOllama(**model_config)


# 2. Tool: test candidate code and return actual observations.
@tool
def test_code(code: str) -> str:
    """Test a complete average(values) function. Pass Python, not Markdown."""
    results = []

    for values, expected in tests:
        actual, error = None, None
        try:
            namespace = {}
            exec(code, namespace)
            actual = namespace['average'](values.copy())

            if type(actual) not in (int, float) or not isfinite(actual):
                raise ValueError('Expected a finite number.')

            passed = isclose(actual, expected, rel_tol=1e-9, abs_tol=1e-12)
        except Exception as exc:
            actual, passed = None, False
            error = f'{type(exc).__name__}: {exc}'

        results.append(dict(
            input=values, expected=expected, actual=actual,
            passed=passed, error=error,
        ))

    return json.dumps({
        'all_passed': all(row['passed'] for row in results),
        'results': results,
    })


# 3. Agent: propose code -> test -> observe -> revise.
system_prompt = (
    'You repair Python functions using test_code. '
    'First test the original code. Then revise it using the test feedback. '
    'Make one tool call at a time. Stop when all tests pass. '
    'The delivered repair is the last code submitted to test_code. '
    'Finish with a short status, not another code version. '
    'Never invent test results.'
)

agent = create_agent(
    model=chat,
    tools=[test_code],
    middleware=[
        ModelCallLimitMiddleware(run_limit=6, exit_behavior='error'),
    ],
)


# 4. Rollout: collect the actual message history.
# Include the system message explicitly so it is retained in the history.
messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=(
        'Fix average(values). Keep its name and argument. '
        'Return 0.0 for empty input; otherwise return the arithmetic mean.\n'
        f'Original code:\n{buggy_code}'
    )),
]

run_error, printed = None, 0

try:
    for state in agent.stream(
        {'messages': messages},
        config={'recursion_limit': 50},
        stream_mode='values',
    ):
        # Each snapshot already contains the accumulated messages.
        messages = list(state['messages'])

        for message in messages[printed:]:
            message.pretty_print()
        printed = len(messages)

except Exception as exc:
    # Preserve the partial trajectory when execution fails.
    run_error = f'{type(exc).__name__}: {exc}'


# 5. Outcome: inspect the tool result, not the assistant's claim.
test_messages = [
    m for m in messages
    if isinstance(m, ToolMessage) and m.name == 'test_code'
]
last_test = test_messages[-1] if test_messages else None

try:
    report = json.loads(last_test.content) if last_test else {}
except (TypeError, json.JSONDecodeError):
    report = {}


# 6. Dataset: one JSONL line = one rollout, including unsuccessful runs.
trajectory = {
    'format': 'langchain_raw_v1',
    'task_id': 'average_v1',
    'run_id': uuid4().hex,
    'model_config': model_config,
    'tests': tests,
    'tools': [{
        'type': 'function',
        'function': {
            'name': test_code.name,
            'description': test_code.description,
            'parameters': test_code.args_schema.model_json_schema(),
        },
    }],
    'messages': [m.model_dump(mode='json') for m in messages],
    'outcome': {
        'last_test_call_id': last_test.tool_call_id if last_test else None,
        'last_test_passed': report.get('all_passed', False),
        'run_error': run_error,
    },
}

log_path = Path('react_trajectories.jsonl')

with log_path.open('a', encoding='utf-8') as file:
    file.write(json.dumps(trajectory, ensure_ascii=False) + '\n')

print('Outcome:', trajectory['outcome'])
print('Saved:', log_path.resolve())